# 02 — Tratamento e Feature Engineering
**Projeto:** churn-upsell-analytics · **Etapa 2 de 3**

Aqui criamos as variáveis (features) que vão alimentar a análise de churn
e o scoring de upsell — as mesmas categorias usadas no SQL do projeto.


## 1. Imports e carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent
RAW_PATH     = PROJECT_ROOT / "data" / "raw"       / "Churn_Modelling.csv"
CLEAN_PATH   = PROJECT_ROOT / "data" / "processed" / "customers_clean.csv"

df = pd.read_csv(RAW_PATH)
print(f"Carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

## 2. Padronizar nomes de coluna

In [ ]:
# CustomerId → customer_id, NumOfProducts → num_of_products, etc.
df.columns = (
    df.columns
      .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)  # CamelCase → snake_case
      .str.lower()
)
print(df.columns.tolist())

## 3. Remover colunas sem valor analítico

In [ ]:
# row_number, customer_id e surname são identificadores — não ajudam a explicar churn
# Mantemos customer_id apenas como referência, removemos row_number e surname

cols_to_drop = [c for c in ["row_number", "surname"] if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"Colunas removidas: {cols_to_drop}")
print(f"Colunas restantes: {df.columns.tolist()}")

## 4. Faixa etária (age_segment)

In [ ]:
# Mesma lógica usada no SQL (Query 1) — categorização por idade
# pd.cut() divide um range contínuo em buckets rotulados

df["age_segment"] = pd.cut(
    df["age"],
    bins=[0, 34, 49, 120],
    labels=["Young (18-34)", "Adult (35-49)", "Senior (50+)"]
)
print(df["age_segment"].value_counts().sort_index().to_string())

## 5. Faixa de saldo (balance_segment)

In [ ]:
# Mesma lógica do SQL: Zero, Low, Mid, High
df["balance_segment"] = pd.cut(
    df["balance"],
    bins=[-0.01, 0, 50000, 150000, float("inf")],
    labels=["Zero Balance", "Low Balance", "Mid Balance", "High Balance"]
)
print(df["balance_segment"].value_counts().to_string())

## 6. Quartil de saldo (para scoring de upsell)

In [ ]:
# Equivalente ao NTILE(4) do SQL — divide em 4 grupos iguais por saldo
# qcut() = "quantile cut" = divide em grupos de tamanho igual (não por valor fixo)

df["balance_quartile"] = pd.qcut(
    df["balance"].rank(method="first"),  # rank quebra empates (muitos clientes têm saldo=0)
    q=4,
    labels=[1, 2, 3, 4]
)
print("Distribuição por quartil (deve ser ~2500 em cada):")
print(df["balance_quartile"].value_counts().sort_index().to_string())

## 7. Segmento de upsell — replicando a lógica SQL em Python

In [ ]:
# Mesma regra de negócio da Query 3 do SQL:
# High Upsell = ativo, 1 produto, quartil de saldo 1 ou 2 (top 50%)
# Medium Upsell = ativo, 1 produto, mas fora do top 50%
# Low Upsell = qualquer outro caso (já tem 2+ produtos, ou está inativo)

def calcular_upsell_segment(row):
    if row["exited"] == 1:
        return "N/A (Churned)"
    if row["num_of_products"] == 1 and row["is_active_member"] == 1:
        if row["balance_quartile"] in [1, 2]:
            return "High Upsell Potential"
        else:
            return "Medium Upsell Potential"
    return "Low Upsell Potential"

df["upsell_segment"] = df.apply(calcular_upsell_segment, axis=1)

print("Distribuição de segmentos de upsell:")
print(df["upsell_segment"].value_counts().to_string())

## 8. Tenure bucket (tempo de casa)

In [ ]:
# Clientes muito novos (0-1 ano) ou muito antigos (9-10 anos) tendem a
# ter comportamentos de churn diferentes de clientes "no meio da jornada"

df["tenure_bucket"] = pd.cut(
    df["tenure"],
    bins=[-1, 1, 4, 7, 10],
    labels=["New (0-1y)", "Growing (2-4y)", "Established (5-7y)", "Loyal (8-10y)"]
)
print(df["tenure_bucket"].value_counts().sort_index().to_string())

## 9. Verificação final e exportação

In [ ]:
print("Nulos no dataset final:", df.isnull().sum().sum())
print(f"Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print()
print("Novas colunas criadas neste notebook:")
for c in ["age_segment", "balance_segment", "balance_quartile", "upsell_segment", "tenure_bucket"]:
    print(f"  ✓ {c}")

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"\nSalvo em: {CLEAN_PATH}")
print()
print("=" * 50)
print("  Notebook 02 concluído.")
print("  Próximo: abra 03_analise.ipynb")
print("=" * 50)